# mask-to-mri — Synthetic MRI Generation with pix2pix

Full pipeline: data exploration → training → evaluation.

**Experiment A:** Train pix2pix to generate realistic MRI from masks.  
**Experiment B:** Measure segmentation improvement with synthetic data.

## 0 — Colab Setup (skip if running locally)

In [ ]:
# === RUN THIS CELL ONLY ON COLAB ===
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

os.chdir('/content')
!git clone https://github.com/AmineAitLaamim/Mask-to-MRI
%cd Mask-to-MRI

# Copy dataset from Drive
shutil.copytree(
    '/content/drive/MyDrive/mask-to-mri/dataset/lgg-mri-segmentation',
    'data/raw/lgg-mri-segmentation'
)

# Symlink outputs to Drive (survives disconnects)
os.makedirs('/content/drive/MyDrive/mask-to-mri/outputs', exist_ok=True)
for d in ['checkpoints', 'samples', 'metrics']:
    src = f'outputs/{d}'
    dst = f'/content/drive/MyDrive/mask-to-mri/outputs/{d}'
    os.makedirs(dst, exist_ok=True)
    if os.path.islink(src): os.unlink(src)
    os.symlink(dst, src, target_is_directory=True)

!pip install -q -r requirements.txt

import torch
print(f"GPU: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0)})")

## 1 — Config & Imports

In [ ]:
import sys
sys.path.insert(0, '.')

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

with open('config.yaml') as f:
    config = yaml.safe_load(f)

print("Configuration:")
print(yaml.dump(config, default_flow_style=False))

from src.utils import fix_seed, get_device
fix_seed(config['data']['seed'])
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Device: {device}")

## 2 — Data Exploration

In [ ]:
from src.dataset import get_patient_file_list
import tifffile

raw_dir = config['data']['raw_dir']
patient_data = get_patient_file_list(raw_dir)

print(f"Total patients: {len(patient_data)}")
print(f"Sample: {list(patient_data.keys())[:5]}")

total_slices = sum(len(v) for v in patient_data.values())
slices_per = [len(v) for v in patient_data.values()]
print(f"Total slices: {total_slices}")
print(f"Per patient: min={min(slices_per)}, max={max(slices_per)}, avg={np.mean(slices_per):.1f}")

In [ ]:
# Visualize a sample pair
first_patient = list(patient_data.keys())[0]
img_path, mask_path = patient_data[first_patient][10]

image = tifffile.imread(img_path)
mask = tifffile.imread(mask_path)

print(f"Patient: {first_patient}")
print(f"Image shape: {image.shape}, dtype: {image.dtype}")
print(f"Mask shape: {mask.shape}, dtype: {mask.dtype}")
print(f"Channel stats:")
for i, ch in enumerate(['R/T1', 'G/FLAIR', 'B/T2']):
    print(f"  {ch}: min={image[:,:,i].min()}, max={image[:,:,i].max()}, mean={image[:,:,i].mean():.1f}")
print(f"Mask values: {np.unique(mask)}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title('MRI (3 sequences)')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Tumor Mask')
axes[1].axis('off')

overlay = image.copy()
overlay[mask > 0] = [255, 0, 0]
axes[2].imshow(overlay)
axes[2].set_title('MRI + Overlay')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## 3 — DataLoaders

In [ ]:
from src.dataset import build_dataloaders

loaders = build_dataloaders(
    raw_dir=config['data']['raw_dir'],
    image_size=config['data']['image_size'],
    batch_size=config['training']['batch_size'],
    num_workers=2 if torch.cuda.is_available() else 0,
    seed=config['data']['seed'],
)

In [ ]:
mask, img = next(iter(loaders['train']))
print(f"Mask: {tuple(mask.shape)}  Image: {tuple(img.shape)}")
print(f"Mask range: [{mask.min():.2f}, {mask.max():.2f}]")
print(f"Image range: [{img.min():.2f}, {img.max():.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
mask_vis = ((mask[0, 0].numpy() + 1.0) * 127.5).astype(np.uint8)
img_vis = ((img[0].numpy().transpose(1, 2, 0) + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
axes[0].imshow(mask_vis, cmap='gray')
axes[0].set_title('Mask')
axes[0].axis('off')
axes[1].imshow(img_vis)
axes[1].set_title('MRI')
axes[1].axis('off')
plt.show()

## 4 — Models

In [ ]:
from src.generator import create_generator
from src.discriminator import create_discriminator
from src.utils import print_model_summary

m = config['model']
G = create_generator(
    in_channels=m['input_channels'],
    out_channels=m['output_channels'],
    num_filters=m['num_filters'],
    norm=m['norm'],
).to(device)

D = create_discriminator(
    in_channels=m['input_channels'] + m['output_channels'],
    num_filters=m['num_filters'],
).to(device)

print_model_summary('Generator (U-Net)', G)
print_model_summary('Discriminator (PatchGAN)', D)

In [ ]:
# Verify forward pass
with torch.no_grad():
    fake = G(mask.to(device))
    d_real = D(mask.to(device), img.to(device))
    d_fake = D(mask.to(device), fake)
print(f"G output: {tuple(fake.shape)}")
print(f"D output: {tuple(d_real.shape)}")

## 5 — Training

In [ ]:
from src.train import train

# Auto-resumes from latest checkpoint if one exists
history = train(
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    generator=G,
    discriminator=D,
    config=config,
    device=device,
    checkpoint_dir=config['paths']['checkpoints'],
    samples_dir=config['paths']['samples'],
)

In [ ]:
from src.utils import plot_loss_curves
plot_loss_curves(history, save_path='outputs/samples/loss_curves.png')

## 6 — Generate Synthetic MRI

In [ ]:
import os, glob
from PIL import Image
from src.train import load_checkpoint

checkpoints = sorted(glob.glob('outputs/checkpoints/checkpoint_epoch_*.pt'))
if checkpoints:
    epoch = load_checkpoint(checkpoints[-1], G, D)
    print(f"Loaded epoch {epoch}")

G.eval()
syn_dir = config['data']['synthetic_dir']
os.makedirs(syn_dir, exist_ok=True)

count = 0
with torch.no_grad():
    for mask, real in loaders['test']:
        fake = G(mask.to(device))
        fake_np = ((fake[0].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        real_np = ((real[0].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        Image.fromarray(fake_np).save(os.path.join(syn_dir, f'fake_{count:04d}.png'))
        Image.fromarray(real_np).save(os.path.join(syn_dir, f'real_{count:04d}.png'))
        count += 1
print(f"Saved {count} pairs → {syn_dir}")

## 7 — Evaluation (Exp A)

In [ ]:
from src.evaluate import compute_ssim_batch, compute_psnr_batch, save_eval_results

all_ssim, all_psnr = [], []
G.eval()
with torch.no_grad():
    for mask, real in loaders['test']:
        fake = G(mask.to(device))
        all_ssim.append(compute_ssim_batch(fake, real))
        all_psnr.append(compute_psnr_batch(fake, real))

mean_ssim = np.mean(all_ssim)
mean_psnr = np.mean(all_psnr)
print(f"SSIM: {mean_ssim:.4f}")
print(f"PSNR: {mean_psnr:.2f} dB")

In [ ]:
# Save metrics
metrics = {
    'ssim': round(mean_ssim, 4),
    'psnr': round(mean_psnr, 2),
}
save_eval_results(metrics, metrics_dir=config['paths']['metrics'], prefix='eval_exp_a')

## 8 — Experiment B (Segmentation)

In [ ]:
import segmentation_models_pytorch as smp
from tqdm import tqdm
from src.losses import DiceBCELoss
from src.evaluate import compute_dice_score

def train_segmentation(train_loader, val_loader, epochs=50):
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation="sigmoid",
    ).to(device)
    criterion = DiceBCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for ep in range(1, epochs + 1):
        model.train()
        loss_sum = 0
        for img, mask in tqdm(train_loader, desc=f"Seg {ep}/{epochs}", leave=False):
            img, mask = img.to(device), mask.to(device)
            mask_bin = (mask + 1.0) / 2.0
            pred = model(img)
            loss = criterion(pred, mask_bin)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        if ep % 10 == 0:
            print(f"  Ep {ep}: loss={loss_sum/len(train_loader):.4f}")

    model.eval()
    dices = []
    with torch.no_grad():
        for img, mask in val_loader:
            pred = model(img.to(device))
            pred_b = (pred.cpu().numpy() > 0.5).astype(np.float32)
            mask_b = ((mask + 1.0) / 2.0).numpy()
            dices.append(compute_dice_score(pred_b, mask_b))
    return np.mean(dices)

print("To run Experiment B, uncomment the cells below.")

In [ ]:
# dice_real = train_segmentation(loaders['train'], loaders['test'], epochs=50)
# print(f"Dice (real only): {dice_real:.4f}")
#
# dice_combined = train_segmentation(loaders['train'], loaders['test'], epochs=50)  # with synthetic
# print(f"Dice (real+synth): {dice_combined:.4f}")

## 9 — Final Results

In [ ]:
print("=" * 50)
print("mask-to-mri — Results")
print("=" * 50)
print(f"SSIM: {mean_ssim:.4f}")
print(f"PSNR: {mean_psnr:.2f} dB")
print(f"Synthetic samples: {count}")
print("=" * 50)